<a href="https://colab.research.google.com/github/cy6erlizard/landsat-lake-clarity-adirondack/blob/main/notebooks/04_train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 4 - Train the regional model, then attack it

Every split is a `GroupKFold` on `lagoslakeid`. A lake appears in exactly one test fold
and nothing about it is visible to the model that predicts it. With most of the variance
sitting between lakes, a random split would let the forest recognise a lake it has
already seen and report a score it cannot reproduce anywhere.

The output that matters is **F20**: the distribution of within-lake correlation across
held-out lakes, for the national model and the regional one, with the client's r = -0.22
drawn on it.

In [ ]:
# Cell 1 of 6: repo + Drive
import os, pathlib, subprocess, sys

REPO = "https://github.com/cy6erlizard/landsat-lake-clarity-adirondack.git"
ROOT = pathlib.Path("/content/landsat-lake-clarity-adirondack")
if ROOT.exists():
    subprocess.run(["git", "-C", str(ROOT), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO, str(ROOT)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(ROOT)], check=True)
sys.path.insert(0, str(ROOT / "src"))  # make lakeclarity importable in the running kernel

from google.colab import drive
drive.mount("/content/drive")
os.environ["LAKECLARITY_DATA"] = "/content/drive/MyDrive/lake-clarity"

In [ ]:
# Cell 2 of 6: cross-validate with lakes held out whole
import joblib
import pandas as pd
from lakeclarity import config, features, model, viz

viz.use_style()
train = pd.read_parquet(config.INTERIM_DIR / "train.parquet")

cv = model.grouped_cv(train, n_splits=5)
metrics = cv.pooled_metrics()
for k, v in metrics.items():
    print(f"{k:>18}: {v:,.4f}" if isinstance(v, float) else f"{k:>18}: {v:,}")

cv.fold_scores.to_csv(config.TABLE_DIR / "T08_fold_scores.csv", index=False)
cv.frame.to_parquet(config.INTERIM_DIR / "oof_predictions.parquet", index=False)
print("\nfold-to-fold R2 spread:", cv.fold_scores["r2_log"].round(3).tolist())

In [ ]:
# Cell 3 of 6: THE HEADLINE. Pooled skill is not per-lake skill.
per_lake = model.per_lake_skill(cv, min_obs=8)
per_lake.to_csv(config.TABLE_DIR / "T09_per_lake_skill.csv", index=False)

print(f"pooled Pearson r across all held-out matchups : {metrics['pearson_r_pooled']:+.3f}")
print(f"median WITHIN-lake r across held-out lakes     : {per_lake['r'].median():+.3f}")
print(f"fraction of held-out lakes with negative r     : {(per_lake['r'] < 0).mean():.0%}")
print(f"lakes evaluated                                : {len(per_lake)}")
print()
print("If the second number is far below the first, the pooled score is carried by")
print("between-lake variance and says nothing about tracking one lake through time.")
print("That is the gap the client fell into, and r = -0.22 on Squam is a sample from it.")

In [ ]:
# Cell 4 of 6: the national model, evaluated on identical rows
# LAGOS_US_LANDSAT_Predictions_v1_QAQC is 7.5 GB. Stream it, keep our lakes only.
from lakeclarity import edi

national_path = config.INTERIM_DIR / "national_predictions_region.parquet"
if not national_path.exists():
    src = edi.download("predictions")
    edi.stream_filter(src, national_path, keep_ids=train["lagoslakeid"].unique())

national = pd.read_parquet(national_path)
national["SENSING_TIME"] = pd.to_datetime(national["SENSING_TIME"], format="ISO8601", utc=True)

# Join the national prediction onto the same observations our model was scored on.
obs = cv.frame.copy()
obs["key"] = obs["lagoslakeid"].astype(str) + "_" + obs["year"].astype(str) + "_" + obs["doy"].astype(str)
national["year"] = national["SENSING_TIME"].dt.year
national["doy"] = national["SENSING_TIME"].dt.dayofyear
national["key"] = national["lagoslakeid"].astype(str) + "_" + national["year"].astype(str) + "_" + national["doy"].astype(str)

nat = obs.merge(national[["key", "Secchi_predicted", "QAQC_recommend"]], on="key", how="inner")
nat = nat[nat["QAQC_recommend"].astype(str).str.upper() == "TRUE"]
nat = nat.rename(columns={"Secchi_predicted": "y_pred_m"})
print(f"{len(nat):,} observations have both a regional and a national prediction")

comparison = model.compare_to_national(cv.frame, nat)
comparison.to_csv(config.TABLE_DIR / "T10_per_lake_comparison.csv", index=False)
print(comparison.groupby("model")["r"].describe().round(3).to_string())

In [ ]:
# Cell 5 of 6: importance, on a model that has seen every training lake
X = features.feature_matrix(train)
y = train[config.LOG_TARGET]

rf_full = model.fit_random_forest(X, y)
joblib.dump(rf_full, config.PROCESSED_DIR / "regional_rf.joblib")

imp = model.importance_comparison(rf_full, X, y, n_repeats=10)
imp.to_csv(config.TABLE_DIR / "T11_importance.csv", index=False)
print(imp.head(12).to_string(index=False))
print("\nGini and permutation will disagree. The 15 ratios are algebraic functions of 6")
print("medians, and Gini importance is biased toward correlated predictors. Trust the")
print("permutation column, and show both.")

In [ ]:
# Cell 6 of 6: figures F15 to F20
top_feature = imp.iloc[0]["feature"]

for fig, fid, slug in [
    (model.fig_fold_scores(cv), "F15", "fold_scores"),
    (model.fig_observed_vs_predicted(cv), "F16", "observed_vs_predicted"),
    (model.fig_residual_grid(cv), "F17", "residual_grid"),
    (model.fig_importance(imp), "F18", "importance"),
    (model.fig_partial_dependence(rf_full, X, top_feature), "F19", "partial_dependence"),
    (model.fig_per_lake_skill(comparison), "F20", "per_lake_skill"),
]:
    print("wrote", viz.save(fig, fid, slug).name)